# 붓꽃(iris)의 품종 분류

In [1]:
# '꽃잎과 꽃받침의 크기' 를 기반으로 붓꽃의 품종 분류

# 데이터셋 : iris.csv   

# 컬럼
# SepalLength  (꽃받침의 길이)
# SepalWidth   (꽃받침의 폭)
# PetalLength  (꽃잎의 길이)
# PetalWidth   (꽃임이 너비)

# Name     품종명 (Species)⭐
#     "Iris-setosa", "Iris-versicolor", "Iris-virginica"  세가지 품종

# 출처 : https://www.kaggle.com/uciml/iris

In [2]:
# CSV 파일에는 약 150개의 데이터가 있는데,  
# 100개는 학습(train)을 위해 사용,   50개는 테스트(test)를 위해 사용

In [ ]:
""" 
[수행 단계]

① 필요한 import 수행

② 데이터 읽어오기
   CSV -> DataFrame (변수명 df)
   기초 통계랑 확인
   
③ 입력데이터와 타겟데이터 분리
   입력데이터 →  변수명 df_data
   타겟데이터 →  변수명 target

④ 전처리 (표준화)
   스케일러 객체 변수 → 변수명 scaler

⑤ train, test 세트 분리. 
   - 8:2로 분류 
   - 클래스별로 균등하게 분류되게 하고 
  
⑥ 최적의 하이퍼 파라미터 찾기
   SVC 의 최적 하이퍼 파라미터 찾기 수행.  GridSearchCV 사용

   param_grid = {'C': [0.1, 1, 10, 100],               
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

   최적의 모델 저장 -> 변수명 clf

⑦ test 점수 확인
    
⑧ 다른 평가지표들 확인

⑨ 예측 동작 확인

⑩ 모델 & 스케일러 저장하기
   모델 -> 파일명 iris_model.pkl
   스케일러 -> 파일명 iris_scaler.pkl

   (위 저장 정보는 웹 애플리케이션에서 사용될것임)

⑪ 저장된 모델 & 스케일러 불러오기

⑫ 예측 함수 만들어 보고 동작 시키기

    # 예측 함수 작성
    # 입력값: 웹에서 사용자가 입력한 값
    # 출력값: 분류 문자열 (ex: Iris-setosa, Iris-versicolor, Iris-virginica)
    def predict_iris(sepal_length, sepal_width, petal_length, petal_width) -> str:


★ 각 수행 단계별로 적절한 제목으로 작성
★ 각 수행 단계마다 이 단계가 무엇을 하는 단계이고, 사용하는 파라미터와
   (필요한 경우) 어떠한 입력으로 어떠한 결과가 나오는지 확인하고 설명을 남기기
★ 각 단계마다 내가 무엇을 확인했는지 코드와 함께 설명 남기기
"""
None

# import

In [3]:
import numpy as np
import pandas as pd
import os, joblib
from sklearn.svm import SVC # 모델은 SVC 를 사용합니다.
from sklearn.model_selection import GridSearchCV

# 데이터 읽기

In [4]:
base_path = r'K:\dataset\ML2018\iris'

In [5]:
df = pd.read_csv(os.path.join(base_path, "iris.csv"))
df.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [6]:
df.describe() 

# 4개의 feature 의 scale 이 서로 다르다 -> 전처리 필요.
# 대체로 balanced 하고 outlier 없는 정돈된 데이터

,SepalLength,SepalWidth,PetalLength,PetalWidth
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## 입력데이터, 타겟레이블 분리

In [7]:
# 입력 데이터 준비
df_data = df[["SepalLength", "SepalWidth", "PetalLength", "PetalWidth"]]
df_data.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [8]:
# 타겟 데이터(레이블) 준비
target = df["Name"]
target.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: Name, dtype: object

In [9]:
target.value_counts()

# ↓ 각 클래스별로 균등하게 분포

Name
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

# 전처리

In [10]:
from sklearn.preprocessing import StandardScaler

In [11]:
scaler = StandardScaler()
scaler.fit(df_data)
data_scaled = scaler.transform(df_data)

# 전처리 결과 데이터 프레임으로 확인
df_data = pd.DataFrame(data=data_scaled, columns=df_data.columns)
df_data

,SepalLength,SepalWidth,PetalLength,PetalWidth
0,-0.900681,1.032057,-1.341272,-1.312977
1,-1.143017,-0.124958,-1.341272,-1.312977
2,-1.385353,0.337848,-1.398138,-1.312977
3,-1.506521,0.106445,-1.284407,-1.312977
4,-1.021849,1.263460,-1.341272,-1.312977
...,...,...,...,...
145,1.038005,-0.124958,0.819624,1.447956
146,0.553333,-1.281972,0.705893,0.922064
147,0.795669,-0.124958,0.819624,1.053537
148,0.432165,0.800654,0.933356,1.447956


# train & test 세트 분리

In [12]:
from sklearn.model_selection import train_test_split
# 우선 '훈련세트' '테스트세트' 를 나눈다   8:2
# 타겟 클래스가 동일한 비율로 섞일수 있도록 stratify=  제공
train_input, test_input, train_target, test_target = train_test_split(
    data_scaled, target, test_size=0.2, stratify=target, random_state=42)

train_input.shape, test_input.shape

((120, 4), (30, 4))

In [13]:
# ↓ 동일한 비율로 섞였는지 확인
train_target.value_counts()

Name
Iris-setosa        40
Iris-virginica     40
Iris-versicolor    40
Name: count, dtype: int64

In [14]:
test_target.value_counts()

Name
Iris-setosa        10
Iris-virginica     10
Iris-versicolor    10
Name: count, dtype: int64

# 최적의 파라미터 찾기

In [15]:
# SVS 의 최척 파라미터
param_grid = {'C': [0.1, 1, 10, 100],               
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

In [16]:
gs = GridSearchCV(SVC(), param_grid, cv=5)

In [17]:
gs.fit(train_input, train_target)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'gamma': [0.001, 0.1, ...], 'kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the

In [18]:
gs.best_params_  # 최적 파라미터 확인

{'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}

In [19]:
clf = gs.best_estimator_  # 최적 파라미터로 학습된 모델
clf

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


# 점수

In [20]:
# 테스트 점수 확인
clf.score(test_input, test_target)

0.9666666666666667

In [21]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report


In [22]:
pred = clf.predict(test_input)  # 테스트 입력의 예측값

In [23]:
accuracy_score(test_target, pred)

0.9666666666666667

In [24]:
recall_score(test_target, pred, average=None)

array([1. , 0.9, 1. ])

In [25]:
precision_score(test_target, pred, average=None)

array([1.        , 1.        , 0.90909091])

In [26]:
f1 = f1_score(test_target, pred, average=None)
print("F1 Score: {}".format(f1))

F1 Score: [1.         0.94736842 0.95238095]


In [27]:
print(classification_report(test_target, pred))

                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



In [ ]:
# 평이한 데이터라 매우 높은 점수.

# 예측하기 동작 확인

In [28]:
df.Name.value_counts()

Name
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [29]:
# 특정 샘플 입력을 예측해보자.
# Iris-setosa 확인
df_setosa = df[df.Name == "Iris-setosa"]
df_setosa

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
5,5.4,3.9,1.7,0.4,Iris-setosa
6,4.6,3.4,1.4,0.3,Iris-setosa
7,5.0,3.4,1.5,0.2,Iris-setosa
8,4.4,2.9,1.4,0.2,Iris-setosa
9,4.9,3.1,1.5,0.1,Iris-setosa


In [30]:
df_setosa.loc[40]  # 특정 샘플

SepalLength            5.0
SepalWidth             3.5
PetalLength            1.3
PetalWidth             0.3
Name           Iris-setosa
Name: 40, dtype: object

In [31]:
df.columns

Index(['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Name'], dtype='object')

In [32]:
input_data = df_setosa.loc[[40]][['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']]
input_data 

,SepalLength,SepalWidth,PetalLength,PetalWidth
40,5.0,3.5,1.3,0.3


In [33]:
# 스케일링후에 예측해야 한다
input_scaled = scaler.transform(input_data)
input_scaled 

array([[-1.02184904,  1.03205722, -1.39813811, -1.18150376]])

In [34]:
# 예측결과
clf.predict(input_scaled)[0]

'Iris-setosa'

# 모델 & 스케일 정보 저장하기 & 불러오기

In [35]:
model_path = os.path.join('iris_model.pkl')

In [36]:
joblib.dump(clf, model_path)

['iris_model.pkl']

In [37]:
scaler_path = r'iris_scaler.pkl'

In [38]:
joblib.dump(scaler, scaler_path)

['iris_scaler.pkl']

In [39]:
# 저장된 모델과 스케일러를 다시 불러들이기 위해 리셋.
clf = None
scaler = None

In [40]:
clf = joblib.load(model_path)
scaler = joblib.load(scaler_path)

In [41]:
# 점수 확인
clf.score(test_input, test_target)  # 이전과 동일한지 확인

0.9666666666666667

In [42]:
# 입력값 준비하여 예측해보자.
sepal_length, sepal_width, petal_length, petal_width = 5.0, 3.5, 1.3, 0.3

In [43]:
df_data.columns

Index(['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth'], dtype='object')

In [44]:
pd.DataFrame([[sepal_length, sepal_width, petal_length, petal_width]], columns=df_data.columns)

,SepalLength,SepalWidth,PetalLength,PetalWidth
0,5.0,3.5,1.3,0.3


In [45]:
input_scaled = scaler.transform(pd.DataFrame([[sepal_length, sepal_width, petal_length, petal_width]], columns=df_data.columns))
input_scaled

array([[-1.02184904,  1.03205722, -1.39813811, -1.18150376]])

In [46]:
clf.predict(input_scaled)[0]

'Iris-setosa'

# 예측함수 만들기

In [47]:
# clf 와 scaler 가 있다면

columns = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']

# 예측 함수 작성
# 입력값: 웹에서 사용자가 입력한 값
# 출력값: 분류 문자열 (ex: Iris-setosa, Iris-versicolor, Iris-virginica)
def predict_iris(sepal_length, sepal_width, petal_length, petal_width) -> str:
    input_scaled = scaler.transform(pd.DataFrame([[sepal_length, sepal_width, petal_length, petal_width]], columns=columns))
    return clf.predict(input_scaled)[0]

In [48]:
df.Name.value_counts()

Name
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [49]:
df[df.Name == 'Iris-setosa'].head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [50]:
df[df.Name == 'Iris-versicolor'].head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
50,7.0,3.2,4.7,1.4,Iris-versicolor
51,6.4,3.2,4.5,1.5,Iris-versicolor
52,6.9,3.1,4.9,1.5,Iris-versicolor
53,5.5,2.3,4.0,1.3,Iris-versicolor
54,6.5,2.8,4.6,1.5,Iris-versicolor


In [51]:
df[df.Name == 'Iris-virginica'].head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
100,6.3,3.3,6.0,2.5,Iris-virginica
101,5.8,2.7,5.1,1.9,Iris-virginica
102,7.1,3.0,5.9,2.1,Iris-virginica
103,6.3,2.9,5.6,1.8,Iris-virginica
104,6.5,3.0,5.8,2.2,Iris-virginica


In [52]:
predict_iris(5.0, 3.5, 1.3, 0.3)

'Iris-setosa'

In [53]:
predict_iris(5.5, 2.3, 4.0, 1.3)

'Iris-versicolor'

In [54]:
predict_iris(6.5, 3.0, 5.8, 2.2)

'Iris-virginica'